## Pré-processamento

Aqui os dados minuto a minuto viram as duas tabelas usadas nas próximas etapas:

- **`dias`**: uma linha por pessoa-dia, com a glicose do dia, o comportamento (carboidratos, proteína, gordura, fibra, atividade e horário da última refeição) e os dados do `bio.csv`;
- **`dataset_rna`**: os exemplos de treino da rede, em que a entrada é a glicose dos dias anteriores + o comportamento do dia + o bio, e o alvo é a glicose média do dia.

As decisões da EDA valem aqui: sensor Dexcom, dia válido com cobertura ≥ 90% e exclusão dos participantes 007, 018 e 026.

In [32]:
# configurações
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
# importa as funções do src/dados.py
from src.dados import (carregar, listar_pids, resumo_dia, maior_sequencia, carregar_bio,
                       tratar_estranhos, montar_dataset_rna, montar_blocos,
                       salvar_tabela, carregar_tabela)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Agora testando a validade das funções para confirmar as informações presentes nos dados que iremos processar

In [33]:
# teste das funções
pids = listar_pids()
print(len(pids), pids[:5])

df = carregar(1)
print(df.shape)
df.columns.tolist()

45 [1, 2, 3, 4, 5]
(14730, 16)


['Unnamed: 0',
 'Timestamp',
 'Libre GL',
 'Dexcom GL',
 'HR',
 'Calories (Activity)',
 'METs',
 'Meal Type',
 'Calories',
 'Carbs',
 'Protein',
 'Fat',
 'Fiber',
 'Amount Consumed',
 'Image path',
 'pid']

Os números e os nomes das colunas estão OK.

Agora para o resumo de cada dia

Os arquivos têm uma linha por minuto, mas a rede prevê a glicose de um dia e o AG recomenda o comportamento de cada dia. A função `resumo_dia` transforma os minutos de um participante em uma linha por dia, com a cobertura do Dexcom (para filtrar os dias válidos), a glicose (média, desvio, pico e % acima de 140 mg/dL) e as seis variáveis de comportamento: carboidratos, proteína, gordura, fibra, atividade e horário da última refeição. Usar dias também evita o vazamento dos valores interpolados minuto a minuto. Na atividade, `min_count=1` faz um dia sem registro do Fitbit virar vazio, e não 0.

In [34]:
# a função resumo_dia está em src/dados.py; aqui testamos no participante 001
resumo_dia(df)

,data,cobertura,glic_media,glic_dp,glic_pico,pct_140,carbs,proteina,gordura,fibra,n_refeicoes,atividade,hora_ultima_refeicao,pid
0,2020-05-01,0.514583,119.259109,13.432144,176.0,6.072874,103.0,88.0,54.2,12.0,2,1960.298337,20.800000,1
1,2020-05-02,0.996528,97.637631,17.872912,142.0,0.418118,220.0,43.0,76.5,7.0,5,3297.322870,19.866667,1
2,2020-05-03,0.996528,96.742160,11.871717,132.0,0.000000,182.0,152.0,57.5,6.0,6,3149.183949,22.433333,1
3,2020-05-04,1.000000,104.495833,12.702486,144.0,0.972222,210.0,47.0,74.0,3.0,5,2565.539679,21.783333,1
4,2020-05-05,1.000000,101.417361,9.145802,126.0,0.000000,116.0,120.0,67.0,11.0,4,3060.174796,21.300000,1
5,2020-05-06,1.000000,103.157639,9.100765,129.0,0.000000,191.0,118.0,63.5,6.0,4,2873.349914,20.966667,1
6,2020-05-07,1.000000,109.981250,10.588680,160.0,1.736111,160.0,62.9,39.5,12.0,4,3238.088277,21.833333,1
7,2020-05-08,1.000000,110.192361,10.551206,138.0,0.000000,178.0,161.7,56.5,12.0,5,3483.099349,22.483333,1
8,2020-05-09,1.000000,113.291667,13.405409,155.0,4.861111,137.0,63.0,67.0,12.0,4,3307.387516,18.683333,1
9,2020-05-10,1.000000,112.961806,9.685295,143.0,1.597222,144.0,66.0,30.5,0.0,3,2545.095884,19.983333,1


No participante 001, os 14.730 minutos viram 11 linhas, uma por dia. Os dias da ponta aparecem com cobertura baixa (0,51 em 01/05 e 0,40 em 11/05) e também com somas menores, como no último dia, que tem uma única refeição e só 73 g de carboidrato. Eles ficam de fora pela regra dos 90%. Os 9 dias inteiros têm cobertura perto de 1 e glicose média entre 97 e 113 mg/dL, o esperado para alguém do grupo saudável. Como o dia vai da meia-noite às 23h59, um lanche depois da meia-noite conta para o dia seguinte. Por isso, a `hora_ultima_refeicao` pode subestimar o quanto a pessoa comeu tarde, um limite que aceitamos para manter o dia igual ao da glicose.

### Todos os participantes

A `resumo_dia` foi para o `src/dados.py`, junto com a `maior_sequencia`. Na mudança, ela passou a completar o calendário: uma data sem nenhuma linha no arquivo entra com cobertura 0, para que a contagem de dias seguidos não pule o buraco. Agora ela roda para os 45 participantes, e as regras da EDA (dia bom com cobertura ≥ 90% e pelo menos 9 dias bons seguidos) separam os válidos.

In [35]:
# resumo de todos os 45 participantes
todos = pd.concat([resumo_dia(carregar(p)) for p in listar_pids()], ignore_index=True)

# dia bom = Dexcom com valor em pelo menos 90% dos minutos
todos['bom'] = todos['cobertura'] >= 0.9

# participante válido = pelo menos 9 dias bons seguidos
seq = todos.groupby('pid')['bom'].apply(maior_sequencia)
validos = seq[seq >= 9].index.tolist()
excluidos = seq[seq < 9].index.tolist()
print('excluídos:', excluidos, '| válidos:', len(validos))

excluídos: [7, 18, 26] | válidos: 42


In [36]:
# tabela final de dias, só com os participantes válidos
dias = todos[todos['pid'].isin(validos)].reset_index(drop=True)
print(len(dias), 'dias no calendário,', dias['bom'].sum(), 'dias bons')

# dias bons com algum valor vazio
dias[dias['bom']].isna().sum()[lambda s: s > 0]

493 dias no calendário, 378 dias bons


atividade    1
dtype: int64

O resumo dos 45 participantes chega ao mesmo resultado da EDA: o 007, o 018 e o 026 não têm 9 dias bons seguidos e ficam de fora. A tabela `dias` fica com os 42 válidos, somando 493 dias no calendário, dos quais 378 são bons. Entre esses dias bons, só um tem valor vazio: a atividade do dia em que o participante 033 não usou o Fitbit. Na EDA ele aparecia como 0 caloria, mas agora, com o `min_count=1`, aparece como vazio, o que é mais honesto, porque não sabemos quanto a pessoa se mexeu. Esse dia, junto com os dias com carboidrato muito alto, é tratado na próxima etapa.

### Dados do participante (`bio.csv`)

A rede precisa saber quem é a pessoa, porque a EDA mostrou que cerca de 90% da variação da glicose vem daí. Do `bio.csv`, entram idade, sexo, IMC, HbA1c e glicemia de jejum, as medidas mais ligadas à glicose. Ficaram de fora a insulina, o colesterol e as medidas por picada no dedo: com só 42 participantes, cada coluna a mais é um número a mais para a rede ajustar com poucos exemplos. O sexo vira número (1 = masculino, 0 = feminino), e o grupo clínico é criado pela HbA1c, como na EDA. O grupo serve para analisar os resultados, mas não entra na rede, porque a própria HbA1c já entra.

In [37]:
bio = carregar_bio()

# junta o bio a cada dia, pelo número do participante
# validate='many_to_one' garante que cada participante aparece uma vez só no bio
n_antes = len(dias)
dias = dias.merge(bio, on='pid', how='left', validate='many_to_one')
print('linhas antes:', n_antes, '| depois:', len(dias), '| dias sem bio:', dias['idade'].isna().sum())
bio.head()

linhas antes: 493 | depois: 493 | dias sem bio: 0


,pid,idade,sexo_masc,imc,hba1c,glicemia_jejum,grupo
0,1,27,1,22.265239,5.4,91,saudável
1,2,49,0,30.946742,5.5,93,saudável
2,3,59,0,26.948690,6.5,118,DM2
3,4,33,0,42.384279,5.5,105,saudável
4,5,51,0,30.957534,6.6,144,DM2


In [38]:
# confere: participantes por grupo e glicose média dos dias bons
print(dias.drop_duplicates('pid')['grupo'].value_counts(), '\n')
dias[dias['bom']].groupby('grupo', observed=True)['glic_media'].mean().round(1)

grupo
saudável        14
pré-diabetes    14
DM2             14
Name: count, dtype: int64 



grupo
saudável        120.7
pré-diabetes    136.7
DM2             166.7
Name: glic_media, dtype: float64

A junção não perdeu nem duplicou nenhuma linha e todos os dias receberam os dados do participante. Os 42 válidos ficam com 14 participantes em cada grupo, e a glicose média dos dias bons sobe de 121 mg/dL no grupo saudável para 137 no pré-diabetes e 167 no DM2, os mesmos valores da EDA.

### Valores estranhos

Cinco dias bons têm mais de 500 g de carboidrato (a mediana é 188 g), nos participantes 004, 021 e 036. Remover esses dias tiraria participantes do projeto, porque quebraria a sequência de 9 dias, então o valor foi limitado a 500 g. O dia sem atividade é o 25/04 do participante 033, quando o Fitbit parou de registrar. Ele é o 9º dos 9 dias bons do 033, então removê-lo também excluiria o participante. A atividade desse dia foi preenchida com a média dos outros dias bons do próprio 033. Os picos de 400 mg/dL ficam como estão: é o limite do sensor e só registramos que o valor real pode ter sido maior.

In [39]:
print('dias bons com mais de 500 g de carboidrato:', (dias.loc[dias['bom'], 'carbs'] > 500).sum())

dias = tratar_estranhos(dias, teto_carbs=500)

print('maior valor de carboidrato depois:', dias.loc[dias['bom'], 'carbs'].max())
print('valores vazios nos dias bons:', dias[dias['bom']].isna().sum().sum())
dias.loc[(dias['pid'] == 33) & dias['bom'], ['data', 'atividade']].tail(3)

dias bons com mais de 500 g de carboidrato: 5
maior valor de carboidrato depois: 500.0
valores vazios nos dias bons: 0


,data,atividade
325,2022-04-23,1985.512498
326,2022-04-24,2521.462496
327,2022-04-25,2585.390623


Os 5 dias acima de 500 g passaram a valer 500 g, e o dia 25/04 do 033 ficou com a média de atividade dele nos outros dias bons. Depois disso, nenhum dia bom tem valor vazio.

### Tabela da rede

Cada exemplo de treino é um dia bom *d*. A entrada tem a glicose dos dias anteriores (média, desvio, pico e % acima de 140), o comportamento de *d* e os dados do participante; o alvo é a glicose média de *d*. A glicose do próprio dia *d* nunca entra na entrada, senão a rede estaria vendo a resposta. Um dia só vira exemplo se ele e todos os dias anteriores usados forem bons. Montamos duas versões, com 2 e com 3 dias anteriores: com 3, a rede tem mais histórico, mas cada participante perde um exemplo a mais no começo.

In [40]:
dataset_rna_2 = montar_dataset_rna(dias, n_anteriores=2)
dataset_rna_3 = montar_dataset_rna(dias, n_anteriores=3)

for nome, ds in [('2 dias anteriores', dataset_rna_2), ('3 dias anteriores', dataset_rna_3)]:
    print(f'{nome}: {len(ds)} exemplos, {ds["pid"].nunique()} participantes, '
          f'{ds.shape[1] - 4} variáveis de entrada, valores vazios: {ds.isna().sum().sum()}')
    print('   por grupo:', ds.groupby('grupo', observed=True).size().to_dict())

dataset_rna_2.head()

2 dias anteriores: 294 exemplos, 42 participantes, 19 variáveis de entrada, valores vazios: 0
   por grupo: {'saudável': 98, 'pré-diabetes': 98, 'DM2': 98}
3 dias anteriores: 252 exemplos, 42 participantes, 23 variáveis de entrada, valores vazios: 0
   por grupo: {'saudável': 84, 'pré-diabetes': 84, 'DM2': 84}


,pid,data,grupo,glic_media_ant1,glic_dp_ant1,glic_pico_ant1,pct_140_ant1,glic_media_ant2,glic_dp_ant2,glic_pico_ant2,...,gordura,fibra,atividade,hora_ultima_refeicao,idade,sexo_masc,imc,hba1c,glicemia_jejum,alvo
0,1,2020-05-04,saudável,96.742160,11.871717,132.0,0.000000,97.637631,17.872912,142.0,...,74.0,3.0,2565.539679,21.783333,27,1,22.265239,5.4,91,104.495833
1,1,2020-05-05,saudável,104.495833,12.702486,144.0,0.972222,96.742160,11.871717,132.0,...,67.0,11.0,3060.174796,21.300000,27,1,22.265239,5.4,91,101.417361
2,1,2020-05-06,saudável,101.417361,9.145802,126.0,0.000000,104.495833,12.702486,144.0,...,63.5,6.0,2873.349914,20.966667,27,1,22.265239,5.4,91,103.157639
3,1,2020-05-07,saudável,103.157639,9.100765,129.0,0.000000,101.417361,9.145802,126.0,...,39.5,12.0,3238.088277,21.833333,27,1,22.265239,5.4,91,109.981250
4,1,2020-05-08,saudável,109.981250,10.588680,160.0,1.736111,103.157639,9.100765,129.0,...,56.5,12.0,3483.099349,22.483333,27,1,22.265239,5.4,91,110.192361


As duas versões têm os 42 participantes e nenhum valor vazio. Com 2 dias anteriores são 294 exemplos (7 por participante) e, com 3, são 252 (6 por participante), sempre divididos igualmente entre os grupos. A conta fecha porque cada participante válido tem exatamente 9 dias bons: os 2 ou 3 primeiros só servem de histórico, e o resto vira exemplo. Qual das duas versões funciona melhor vai ser decidido no treino da rede, comparando o erro nos mesmos dias de teste.

### Bloco D1 a D9

O algoritmo genético e a avaliação final trabalham com 9 dias seguidos de cada participante: D1 a D6 são observados e, de D7 a D9, o AG propõe o comportamento. A função `montar_blocos` pega os primeiros 9 dias bons seguidos de cada participante e numera os dias de 1 a 9 na coluna `dia`.

In [41]:
blocos = montar_blocos(dias)

print('blocos:', blocos['pid'].nunique(), '| dias por bloco:', blocos.groupby('pid').size().unique())
print('datas seguidas:', (blocos.groupby('pid')['data'].diff().dropna() == pd.Timedelta(days=1)).all())
print('todos os dias bons:', blocos['bom'].all(), '| valores vazios:', blocos.isna().sum().sum())
print('por grupo:', blocos.drop_duplicates('pid')['grupo'].value_counts().to_dict())

blocos.loc[blocos['pid'] == 1, ['pid', 'dia', 'data', 'glic_media', 'carbs', 'atividade']]

blocos: 42 | dias por bloco: [9]
datas seguidas: True
todos os dias bons: True | valores vazios: 0
por grupo: {'saudável': 14, 'pré-diabetes': 14, 'DM2': 14}


,pid,dia,data,glic_media,carbs,atividade
0,1,1,2020-05-02,97.637631,220.0,3297.322870
1,1,2,2020-05-03,96.742160,182.0,3149.183949
2,1,3,2020-05-04,104.495833,210.0,2565.539679
3,1,4,2020-05-05,101.417361,116.0,3060.174796
4,1,5,2020-05-06,103.157639,191.0,2873.349914
5,1,6,2020-05-07,109.981250,160.0,3238.088277
6,1,7,2020-05-08,110.192361,178.0,3483.099349
7,1,8,2020-05-09,113.291667,137.0,3307.387516
8,1,9,2020-05-10,112.961806,144.0,2545.095884


São 42 blocos de 9 dias, todos bons, em datas seguidas e sem valores vazios, 14 em cada grupo. Como cada participante tem exatamente 9 dias bons, o bloco são os próprios dias bons dele.

### Salvar as tabelas

As tabelas prontas vão para `data/processed/`, que fica fora do git junto com os dados brutos. Assim os próximos notebooks não precisam reler os 45 arquivos minuto a minuto. A função `carregar_tabela` lê de volta com a data e o grupo clínico nos tipos certos, já que um CSV guarda tudo como texto.

In [42]:
tabelas = {
    'dias': dias,
    'blocos_d1_d9': blocos,
    'dataset_rna_2': dataset_rna_2,
    'dataset_rna_3': dataset_rna_3,
}

for nome, tabela in tabelas.items():
    caminho = salvar_tabela(tabela, nome)
    lida = carregar_tabela(nome)      # confere se a tabela volta igual depois de gravada
    print(f'{caminho.name}: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas | '
          f'igual depois de ler: {lida.equals(tabela)}')

dias.csv: 493 linhas x 21 colunas | igual depois de ler: True
blocos_d1_d9.csv: 378 linhas x 22 colunas | igual depois de ler: True
dataset_rna_2.csv: 294 linhas x 23 colunas | igual depois de ler: True
dataset_rna_3.csv: 252 linhas x 27 colunas | igual depois de ler: True


As quatro tabelas foram gravadas e, lidas de volta, ficam idênticas às da memória, com a data e o grupo nos tipos certos. Com isso o pré-processamento termina: os notebooks da rede e do algoritmo genético começam com `carregar_tabela('dataset_rna_2')` ou `carregar_tabela('blocos_d1_d9')`, sem refazer nada daqui.

Um cuidado para a próxima etapa: os dias D7 a D9 de cada participante também aparecem na tabela da rede como exemplos de treino. Na avaliação da recomendação, a rede usada para um participante precisa ter sido treinada sem os dados dele, o que a validação por participante (GroupKFold) já garante.